# Python & NumPy Warm-Up

Computationally, quantum computing is linear algebra over the complex numbers. Every
state you will ever prepare is a vector. Every gate is a matrix. Every measurement
probability is the squared magnitude of one entry of a vector. That is genuinely the
whole of it — but it is unforgiving about notation, and a learner who is still
fighting array syntax has no attention left over for the physics.

So this notebook is deliberately quantum-free. It builds the handful of NumPy habits
that every later notebook leans on, and for each one it names the quantum job that
habit will eventually do. If you already write NumPy fluently, skim the tables, run
the exercises to be sure, and move on — this is a warm-up, not a course in NumPy.

**Objectives:**
- Create vectors and matrices with `np.array`
- Multiply matrices and vectors with `@`
- Take inner products, norms, conjugates, and transposes
- Build tensor (Kronecker) products with `np.kron`
- Work with complex numbers using Python's built-in `1j`

**How to work through it:** run every code cell as you meet it and read the printed
output before moving on. The output is the lesson; the prose only tells you what to
look for.

**Reference:** See [`../GUIDE.md`](../GUIDE.md). **No quantum content yet.**

<!-- browser-runnable -->


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sanity check: this notebook needs only NumPy and Matplotlib.
print(f'NumPy: {np.__version__}')

## 1. Plain English

A **vector** is an ordered list of numbers. A **matrix** is a rectangle of numbers.
Multiplying a matrix by a vector transforms that vector into a new vector — it is a
machine that takes a list in and gives a different list back.

That is the entire vocabulary. What makes it feel like more is that quantum mechanics
attaches a physical meaning to each piece, and the meanings are unusual:

| Linear algebra | What it will mean in a few notebooks |
|---|---|
| A unit vector with complex entries | The state of a quantum system |
| A matrix that preserves length | A quantum gate — something a computer can *do* |
| Squared magnitude of an entry | The probability of seeing that outcome |
| Tensor product of two vectors | Two separate systems considered as one |

Notice the constraint hiding in rows one and two: quantum states are always **unit
length**, and gates are exactly the matrices that keep them that way. That single
rule is why probabilities always sum to 1, and it is the reason so much of what
follows is about norms.

Everything in this notebook is ordinary NumPy. The physics arrives in notebook 4;
what you build here is the fluency to read it without stumbling.


## 2. Code first

The fastest way to learn a notation is to watch it do something. This section runs
through the five operations you will use constantly, in the order you will meet them.
Read the comment on each line, run the cell, and check that the printed answer is the
one you expected before you scroll on.

### Vectors as 1D arrays

NumPy represents a vector as a one-dimensional array: `np.array([3.0, 4.0])`. Two
things to notice about the cell below.

First, `shape` is `(2,)` — a tuple of length one. NumPy distinguishes a flat 2-element
vector, shape `(2,)`, from a 2x1 column matrix, shape `(2, 1)`. Quantum textbooks draw
states as columns, but in code you will use the flat form almost everywhere, and
`np.kron` and `@` both do the right thing with it. A surprising number of confusing
NumPy errors are really a `(2,)` where a `(2, 1)` was expected, so it is worth
checking `.shape` whenever a result looks strange.

Second, `np.linalg.norm(v)` is the ordinary Pythagorean length: $\sqrt{3^2 + 4^2} = 5$.
The vector `(3, 4)` is *not* a legal quantum state, because its length is 5 rather
than 1. Turning an arbitrary vector into a legal state means dividing by its norm,
which you will do dozens of times.


In [ ]:
# A 2-vector with real entries.
v = np.array([3.0, 4.0])
print('vector:', v)
print('shape:', v.shape)
print('length (norm):', np.linalg.norm(v))   # sqrt(3^2 + 4^2) = 5

In [ ]:
# A 2-vector with COMPLEX entries. Python uses '1j' for the imaginary unit.
psi = np.array([1 + 0j, 1j])
print('psi:', psi)
print('dtype:', psi.dtype)

# Conjugate flips the sign of every imaginary part.
print('conjugate:', psi.conj())

Complex entries are where NumPy starts to feel specifically quantum. Python writes the
imaginary unit as `1j`, never `i`, and `1j` alone is a literal — `j` by itself is an
undefined name. The moment a single entry of an array is complex, NumPy promotes the
entire array to `complex128`; you never declare it.

The cell above also calls `.conj()`, which flips the sign of every imaginary part.
That operation looks like busywork now, and it is the single most commonly forgotten
step later on: the length of a complex vector is *not* the square root of the sum of
its squared entries, but the sum of each entry times its own conjugate. Skip the
conjugate and you can compute a "probability" that is negative or complex, which is
the classic first bug in hand-written quantum code.

### Matrices as 2D arrays

A matrix is a list of rows, so it is a nested list handed to `np.array`. The matrix
below happens to be the **X gate** — the quantum NOT — though it is just numbers for
now. Note the transpose `M.T`, which reflects the rectangle across its diagonal. For
complex matrices you will almost always want `M.conj().T` instead, written
$M^\dagger$ and pronounced "M dagger": transpose *and* conjugate. Reaching for `.T`
alone on a complex matrix is the matrix-sized version of the bug in the paragraph
above.


In [ ]:
M = np.array([
    [0, 1],
    [1, 0],
])
print('M:')
print(M)
print('shape:', M.shape)
print('transpose:')
print(M.T)

### The two operators you will use most

- `@` is matrix multiplication. It also covers matrix-times-vector and
  vector-times-vector (which gives the inner product, a single number). Do not use
  `*` — on NumPy arrays `*` multiplies entry by entry, which is a different and
  usually wrong operation that fails silently because the shapes happen to match.
- `np.kron(A, B)` is the **tensor product**, written $A \otimes B$. This is how two
  separate systems are combined into one: tensoring two 2-vectors gives a 4-vector,
  three gives an 8-vector, and $n$ gives a $2^n$-vector.

That exponential is the headline fact of the whole field, and it is worth pausing on.
Each qubit you add doubles the length of the vector needed to describe the system. Ten
qubits need 1,024 numbers; fifty need more than a quadrillion, which is why simulating
a fifty-qubit machine on a classical computer is out of reach and why building the
real thing is worth the trouble.

Read the `np.kron` output carefully. The four basis vectors come out in dictionary
order — `|00>`, `|01>`, `|10>`, `|11>` — each with its single 1 one slot further
along. That ordering is not decoration; it is the convention that tells you which
entry of a state vector corresponds to which measurement outcome, and you will read
state vectors this way for the rest of the curriculum.

An ordering warning worth banking now: `np.kron(A, B)` is not `np.kron(B, A)`. Tensor
products do not commute, and swapping the arguments silently relabels which qubit is
which.


In [ ]:
# Matrix times vector
v = np.array([1, 0])      # the column [1, 0]
X = np.array([[0, 1], [1, 0]])
Xv = X @ v
print('X @ v =', Xv)      # should be [0, 1]

# Matrix times matrix
print('X @ X =')
print(X @ X)              # X is its own inverse, so X @ X = I

In [ ]:
# Tensor product: stick two small vectors together into a bigger one.
zero = np.array([1, 0])
one  = np.array([0, 1])

print('|0> ⊗ |0> =', np.kron(zero, zero))   # 4-vector [1, 0, 0, 0]
print('|0> ⊗ |1> =', np.kron(zero, one))    # [0, 1, 0, 0]
print('|1> ⊗ |0> =', np.kron(one, zero))    # [0, 0, 1, 0]
print('|1> ⊗ |1> =', np.kron(one, one))     # [0, 0, 0, 1]

## 3. Notation

You have now used every piece of machinery in the table below. From here on, when you
meet a symbol on the left in prose or in a formula, reach for the code on the right.

| Math | NumPy | Used for |
|---|---|---|
| Vector $v$ | `np.array([...])` | A quantum state |
| Matrix-vector product $Mv$ | `M @ v` | Applying a gate to a state |
| Conjugate $\bar{z}$ | `z.conj()` | Half of every inner product |
| Transpose $M^T$ | `M.T` | Rarely used alone on complex matrices |
| Conjugate transpose $M^\dagger$ | `M.conj().T` | Reversing a gate; testing unitarity |
| Norm $\lVert v \rVert = \sqrt{\sum_i \lvert v_i \rvert^2}$ | `np.linalg.norm(v)` | Checking a state is legal |
| Tensor product $A \otimes B$ | `np.kron(A, B)` | Combining subsystems |

Two of these carry more weight than the others.

**The norm is a validity check.** A quantum state must satisfy
$\lVert \psi \rVert = 1$, because the squared entries are probabilities and probabilities sum
to one. When a calculation goes wrong, `np.linalg.norm` is the first thing to print:
a norm that has drifted away from 1 tells you the error is upstream, before you waste
time on the physics.

**The dagger is how gates undo themselves.** A matrix is **unitary** when
$M^\dagger M = I$, meaning applying $M$ and then $M^\dagger$ returns you exactly
where you started. Unitary matrices are precisely the ones that preserve length, so
they are precisely the legal quantum gates — and it is why every quantum gate is
reversible, which classical AND and OR are not. Exercise 2 has you verify a special
case of this by hand.


## 4. Exercises

Three exercises to lock in the mechanics. Each has tiered hints -- expand
only what you need -- and a check cell that tells you when you have it.
Worked solutions wait at the bottom of the notebook: attempt first, then peek.

### Exercise 1 — A vector and its norm

Build the 4-vector u = (1, 2, 3, 4) and measure its length.

Define `u` — the vector as a NumPy array — and `u_norm` — its Euclidean
norm. The norm should come out near 5.477.

<details><summary>Hint 1 — nudge</summary>

The norm is the square root of the sum of squared entries. The very first
vector cell in Section 2 computed one — which NumPy function did it call?

</details>
<details><summary>Hint 2 — approach</summary>

Pass a plain Python list of the four entries to `np.array(...)`, then hand
the result to `np.linalg.norm(...)` and store what comes back in `u_norm`.

</details>

In [ ]:
# Exercise 1: Build the vector u = (1, 2, 3, 4) and compute its norm.
# Define: u -- the 4-vector as a NumPy array, and u_norm -- its norm.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert u.shape == (4,), "u should be a flat 4-entry vector, shape (4,)"
    assert abs(u_norm - np.linalg.norm(u)) < 1e-9, (
        "u_norm should be the norm of u itself"
    )
    assert abs(u_norm - 5.477) < 0.01, (
        "check u's entries -- the norm of (1, 2, 3, 4) lands near 5.477"
    )

### Exercise 2 — A complex matrix that undoes itself

Build the 2x2 matrix `Y = [[0, -i], [i, 0]]`, then compute `Y @ Y` and
confirm the product is the identity matrix.

Define `Y` — the matrix — and `Y_squared` — the product of `Y` with itself.

<details><summary>Hint 1 — nudge</summary>

Python spells the imaginary unit `1j` (Section 2's complex-vector cell used
it). The moment one entry of an array is complex, NumPy stores the whole
array as complex — no special setup needed.

</details>
<details><summary>Hint 2 — approach</summary>

Write the matrix as a nested list (rows inside an outer list) passed to
`np.array(...)`, using `1j` for the imaginary entries. Multiply with the
`@` operator, and compare the result against `np.eye(2)`.

</details>

In [ ]:
# Exercise 2: Build Y = [[0, -i], [i, 0]] and verify Y @ Y is the identity.
# Define: Y -- the 2x2 complex matrix, and Y_squared -- the product Y @ Y.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert Y.shape == (2, 2), "Y should be a 2x2 matrix"
    assert np.iscomplexobj(Y), "Y needs complex entries -- remember 1j"
    assert np.allclose(Y.real, 0), (
        "every entry of this Y is purely imaginary (or zero) -- no real parts"
    )
    assert np.allclose(Y_squared, Y @ Y), "Y_squared should be the product Y @ Y"
    assert np.allclose(Y_squared, np.eye(2)), (
        "Y @ Y should come out as the 2x2 identity -- check the off-diagonal signs"
    )

### Exercise 3 — A tensor product that stays unit length

Build the unit vector w = (1, 1)/sqrt(2), then take its tensor product with
itself and confirm the result is still a unit vector.

Define `w` — the normalized 2-vector — and `w_tensor` — the tensor product
of `w` with itself. Print `w_tensor` and its norm; the norm should still
be 1.

<details><summary>Hint 1 — nudge</summary>

Section 2 introduced exactly one NumPy function for tensor products. And
dividing an array by a scalar divides every entry — that is all the
normalization takes.

</details>
<details><summary>Hint 2 — approach</summary>

Build the 2-vector of ones, divide it by `np.sqrt(2)`, then pass the result
twice to `np.kron(...)`. Measure both vectors with `np.linalg.norm(...)`.

</details>

In [ ]:
# Exercise 3: Tensor a unit vector with itself and check the norm stays 1.
# Define: w -- the 2-vector (1, 1)/sqrt(2), and w_tensor -- the tensor
# product of w with itself.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert w.shape == (2,), "w should be a 2-vector"
    assert abs(np.linalg.norm(w) - 1.0) < 1e-9, (
        "w should be a UNIT vector -- did you divide by sqrt(2)?"
    )
    assert w_tensor.shape == (4,), (
        "the tensor product of two 2-vectors is a 4-vector"
    )
    assert abs(np.linalg.norm(w_tensor) - 1.0) < 1e-9, (
        "tensor products of unit vectors stay unit length"
    )
    assert np.allclose(w_tensor, w_tensor[0]), (
        "with equal entries in w, every entry of the tensor product matches"
    )

### Solutions (try first, then expand)

In [ ]:
# --- Exercise 1 ---
u = np.array([1, 2, 3, 4])
u_norm = np.linalg.norm(u)
print(u_norm)                          # ~5.477

# --- Exercise 2 ---
Y = np.array([[0, -1j], [1j, 0]])
Y_squared = Y @ Y
print(Y_squared)                       # [[1, 0], [0, 1]]

# --- Exercise 3 ---
w = np.array([1, 1]) / np.sqrt(2)
w_tensor = np.kron(w, w)
print(w_tensor)
print('norm:', np.linalg.norm(w_tensor))     # 1.0

## Summary

- A quantum state is a **unit** vector; a gate is a **length-preserving** matrix. Both
  facts are enforced by norms, so `np.linalg.norm` is your first debugging tool.
- Python's imaginary unit is `1j`. Any complex entry promotes the whole array.
- `@` multiplies matrices; `*` does not. `.conj().T` (the dagger), not `.T`, is what
  you want for complex matrices.
- `np.kron` combines subsystems, and it is where the $2^n$ growth comes from — the
  fact that makes quantum computers both hard to simulate and worth building.
- Basis vectors come out of `np.kron` in dictionary order, which fixes the meaning of
  every entry in every state vector you will read from here on.

**You finished notebook 1.** If everything above feels routine, continue to
[`02-linear-algebra-for-quantum.ipynb`](02-linear-algebra-for-quantum.ipynb), where the
same operations acquire names — inner product, unitary, eigenvector — and start
earning their keep.
